In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset, DatasetDict
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)

print("Environment ready for mBERT!")

Environment ready for mBERT!


In [ ]:
train_df = pd.read_csv('trainV2.csv')
test_df = pd.read_csv('testV2.csv')
train_df['Class'] = train_df['Class'].str.strip()

mapping = {
    'Non-Abusive': 1,
    'Abusive': 0,
    'abusive': 0
}

train_df['label'] = train_df['Class'].map(mapping)
train_df = train_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)
train_sub_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_sub_df[['Text', 'label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['Text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['Text']].reset_index(drop=True))
})

print("Data distribution in Training set:")
print(train_df['label'].value_counts())
print("\nData loading and cleaning complete.")

Data distribution in Training set:
label
1    1883
0    1769
Name: count, dtype: int64

Data loading and cleaning complete.


In [ ]:
model_name = "bert-base-multilingual-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["Text"], padding="max_length", truncation=True, max_length=128)

# 3. Applokenization to all splits (train, validation, test)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# 4. Remove the raw 'Text' column and set format to PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(["Text"])
tokenized_datasets.set_format("torch")

print("Tokenization for mBERT complete.")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/2921 [00:00<?, ? examples/s]

Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

Tokenization for mBERT complete.


In [ ]:
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

training_args = TrainingArguments(
    output_dir="./mbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("mBERT model and trainer are ready to go!")

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


mBERT model and trainer are ready to go!


In [ ]:
print("Starting mBERT training...")
trainer.train()
final_metrics = trainer.evaluate()
print("\nFinal Validation Metrics:")
print(final_metrics)

Starting mBERT training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.621794,0.597457,0.668947,0.550186,0.919255,0.392573
2,0.466181,0.471009,0.789330,0.787879,0.819484,0.758621
3,0.413775,0.505462,0.787962,0.798440,0.783163,0.814324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Final Validation Metrics:
{'eval_loss': 0.4709596335887909, 'eval_accuracy': 0.7893296853625171, 'eval_f1': 0.7878787878787878, 'eval_precision': 0.8194842406876791, 'eval_recall': 0.7586206896551724, 'eval_runtime': 1.3826, 'eval_samples_per_second': 528.72, 'eval_steps_per_second': 33.271, 'epoch': 3.0}


In [ ]:
print("Generating predictions for mBERT...")
test_results = trainer.predict(tokenized_datasets["test"])
raw_preds = np.argmax(test_results.predictions, axis=-1)
reverse_mapping = {1: 'Non-Abusive', 0: 'Abusive'}
final_labels = [reverse_mapping[p] for p in raw_preds]
final_output_df = pd.DataFrame({
    'Text': test_df['Text'],
    'Class': final_labels
})
output_csv_name = "mbert_test_predictions.csv"
final_output_df.to_csv(output_csv_name, index=False)

print(f"Success! Predictions saved to {output_csv_name}.")
final_output_df.head()

Generating predictions for mBERT...


Success! Predictions saved to mbert_test_predictions.csv.


,Text,Class
0,லக்ஷ்மி அம்மா நீங்க புலம்புங்க அவளுக அவளுகபாட்...,Non-Abusive
1,"இன்னும் கைது பண்ணல... அனைத்து பெற்றோர்களும், க...",Non-Abusive
2,"அப்பா,அம்மா, அந்த இன்டர்வியூ பண்ற வக்கிரம்புடி...",Abusive
3,Suganthi உனக்கு வீட்ல குழந்தையை வச்சிருக்க கார...,Non-Abusive
4,எல்லாமே script thaan. ஷகீலா உங்க scriptum அரும...,Non-Abusive


In [ ]:
# 1. Load both results
# Note: Ensure you have both 'test_predictions.csv' (XLM-R) and
# 'mbert_test_predictions.csv' (mBERT) in your Colab files.
mbert_results = pd.read_csv('mbert_test_predictions.csv')

# 2. Rename columns to distinguish them
xlmr_results = xlmr_results.rename(columns={'Predicted_Class': 'XLM_R_Pred'})
mbert_results = mbert_results.rename(columns={'Class': 'mBERT_Pred'})

# 3. Merge the results on the 'Text' column
comparison_df = pd.merge(xlmr_results, mbert_results, on='Text')

# 4. Calculate agreement
comparison_df['Match'] = comparison_df['XLM_R_Pred'] == comparison_df['mBERT_Pred']
agreement_rate = comparison_df['Match'].mean() * 100

print(f"Model Agreement Rate: {agreement_rate:.2f}%")

# 5. Display samples where the models DISAGREED
print("\nSamples where models disagreed:")
disagreements = comparison_df[comparison_df['Match'] == False]

if not disagreements.empty:
    display(disagreements.head(10))
else:
    print("The models agree on all test samples!")

# 6. Save the comparison report
comparison_df.to_csv('model_comparison_report.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'test_predictions.csv'